In [8]:
%run tardis_eda.ipynb
%tb

SystemExit: 84

# Tardis model : initialisation

- modules calls
- define exit values

In [ ]:
import pandas as pd
from datetime import date
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.tree import plot_tree
from pickle import dump
from enum import Enum
from tardis_tools import get_field_value

TRAIN_SERVICE = Enum("TRAIN_SERVICE", [("National", 1), ("International", 2)])

EPITECH_SUCCESS = 0
EPITECH_FAILURE = 84

MONTH_IN_YEAR = 12

GET_NBR_ROWS = lambda df: df.shape[0]

TEST_SIZE = 20 / 100
RANDOM_STATE = 42
NBR_DECISION_TREES = 200
ARANGE_STEP = 0.01
RESHAPE = (-1, 1)
TREE_FIG_SIZE = (400, 200)
TREE_FONT_SIZE = 10

SHOW = False

# Tardis model

## Tardis model training

#### Convertion from date to duration (date to int) :

arguments :
- csv (panda DataFrame), the dataframe

set the type of the 'Date' column to int\
replace every date by the (date - today) result in month\
return the dataframe

In [11]:
def date_to_int(d: date) -> int:
    today = date.today()
    if type(d) == date:
        return (today.year * MONTH_IN_YEAR + today.month) - (
            d.year * MONTH_IN_YEAR + d.month
        )
    return None


def convert_date_to_duration(csv: pd.DataFrame) -> pd.DataFrame:
    index = (FEATURES_COLUMNS + TARGETS_COLUMNS).index("Date")
    date_data = [
        get_field_value(csv.iloc[[i]]["Date"], date) for i in range(GET_NBR_ROWS(csv))
    ]
    csv["Date"] = pd.to_numeric(csv["Date"], errors="coerce")
    for i in range(GET_NBR_ROWS(csv)):
        csv.iloc[i, index] = date_to_int(date_data[i])
    return csv

#### Convertion from string service to corresponding `TRAIN_SERVICE` enum value (int) :

arguments :
- csv (panda DataFrame), the dataframe

set the type of the 'Service' column to int\
replace every string value by the corresponding int value from the `TRAIN_SERVICE` enum\
return the dataframe

In [12]:
def convert_service(init: str) -> int:
    if init is None:
        return None
    for e in TRAIN_SERVICE:
        if init == e.name:
            return e.value
    return None


def convert_service_column(csv: pd.DataFrame) -> pd.DataFrame:
    index = (FEATURES_COLUMNS + TARGETS_COLUMNS).index("Service")
    str_data = [
        get_field_value(csv.iloc[[i]]["Service"], str) for i in range(GET_NBR_ROWS(csv))
    ]
    csv["Service"] = csv["Service"].str.strip().str.capitalize()
    csv["Service"] = pd.to_numeric(csv["Service"], errors="coerce")
    for i in range(GET_NBR_ROWS(csv)):
        csv.iloc[i, index] = convert_service(str_data[i])
    return csv

#### Convertion from station name (str) to station id (int) :

arguments :
- csv (panda DataFrame), the dataframe

set the type of the 'Departure station' and 'Arrival station' columns to int\
replace every station name (str) by its id (int)\
return the dataframe

In [13]:
def is_a_valid_station(station_name: str) -> bool:
    if type(station_name) != str:
        return False
    return len(station_name) > 0


def set_id_dict(csv: pd.DataFrame) -> dict:
    stations_id = {}
    id = 1
    for i in range(GET_NBR_ROWS(csv)):
        dep = get_field_value(csv.iloc[[i]]["Departure station"], str)
        arr = get_field_value(csv.iloc[[i]]["Arrival station"], str)
        if is_a_valid_station(dep):
            if dep not in stations_id:
                stations_id[dep] = id
                id += 1
        if is_a_valid_station(arr):
            if arr not in stations_id:
                stations_id[arr] = id
                id += 1
    file = open("stations_id.pkl", "w+b")
    dump(stations_id, file)
    return stations_id


def convert_stations_columns(csv: pd.DataFrame) -> pd.DataFrame:
    dep_index = (FEATURES_COLUMNS + TARGETS_COLUMNS).index("Departure station")
    arr_index = (FEATURES_COLUMNS + TARGETS_COLUMNS).index("Arrival station")
    stations_id = set_id_dict(csv)
    get_id = lambda s: stations_id[s] if s in stations_id else None
    dep = []
    arr = []
    for i in range(GET_NBR_ROWS(csv)):
        dep.append(get_field_value(csv.iloc[[i]]["Departure station"], str))
        arr.append(get_field_value(csv.iloc[[i]]["Arrival station"], str))
    csv["Departure station"] = pd.to_numeric(csv["Departure station"], errors="coerce")
    csv["Arrival station"] = pd.to_numeric(csv["Arrival station"], errors="coerce")
    for i in range(GET_NBR_ROWS(csv)):
        csv.iloc[i, dep_index] = get_id(dep[i])
        csv.iloc[i, arr_index] = get_id(arr[i])
    return csv

#### Preparation of the dataset

arguments :
- filename (str), the dataset file name
- separator (str) (',' by default), the separator to use

get the csv file content\
filter uneeded columns\
convert date and string objects to numeric values\
return the dataframe or None in case of error

In [ ]:
def csv_initialization(filename: str, separator: str = ",") -> pd.DataFrame:
    csv = tardis_eda(filename, separator=separator)
    if csv is None:
        return None
    csv = csv[-1]
    csv = convert_date_to_duration(csv)
    csv = convert_service_column(csv)
    route_means = csv.groupby(["Departure station", "Arrival station"])[
        "Average delay of all trains at arrival"
    ].mean()
    dep_means = csv.groupby("Departure station")[
        "Average delay of all trains at arrival"
    ].mean()
    arr_means = csv.groupby("Arrival station")[
        "Average delay of all trains at arrival"
    ].mean()
    csv["route_mean_delay"] = csv.apply(
        lambda row: route_means.loc[row["Departure station"], row["Arrival station"]],
        axis=1,
    )
    csv["dep_mean_delay"] = csv["Departure station"].map(dep_means)
    csv["arr_mean_delay"] = csv["Arrival station"].map(arr_means)
    global_mean = csv["Average delay of all trains at arrival"].mean()
    dump(route_means.to_dict(), open("route_means.pkl", "w+b"))
    dump(dep_means.to_dict(), open("dep_means.pkl", "w+b"))
    dump(arr_means.to_dict(), open("arr_means.pkl", "w+b"))
    dump(global_mean, open("global_mean.pkl", "w+b"))
    csv = convert_stations_columns(csv)
    csv.drop(columns=["Departure station", "Arrival station", "Date"], inplace=True)
    return csv

#### Dataset split

arguments :
- csv (panda Dataframe), the dataframe

split the dataset and return the (features for training, target for training,
features for testing, target for testing) tuple (tuple of 4 numpy ndarray objects)

In [15]:
def split_the_dataset(
    csv: pd.DataFrame, features: np.ndarray, targets: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    features_train, features_test, target_train, target_test = train_test_split(
        features, targets, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    return (features_train, features_test, target_train, target_test)

#### Automatic training

arguments :
- features_train (numpy ndarray)
- target_train (numpy ndarray)
- features_test (numpy ndarray)
- target_test (numpy ndarray)
- regressor (sklearn RandomForestRegressor)

train automatically the model, print its performances and return the array of
its predictions

In [16]:
def automatic_training(
    features_train: np.ndarray,
    targets_train: np.ndarray,
    features_test: np.ndarray,
    targets_test: np.ndarray,
    regressor: ExtraTreesRegressor,
) -> np.ndarray:
    regressor.fit(features_train, targets_train)
    print("Out-of-Bag Score:", regressor.oob_score_)
    predictions = regressor.predict(features_test)
    mae = mean_absolute_error(targets_test, predictions)
    print("Mean Absolute Error:", mae)
    file = open(".model_error", "w")
    print(mae, file=file)
    file.close()
    mse = mean_squared_error(targets_test, predictions)
    print("Mean Squared Error:", mse)
    r2 = r2_score(targets_test, predictions)
    print("R-squared:", r2)
    return predictions

#### Visualization

arguments :
- csv (panda DataFrame)
- regressor (sklearn ExtraTreesRegressor)

In [18]:
def get_tree(csv: pd.DataFrame, regressor: ExtraTreesRegressor):
    tree_to_plot = regressor.estimators_[0]

    plt.figure(figsize=TREE_FIG_SIZE)
    plot_tree(
        tree_to_plot,
        feature_names=csv.columns.to_list(),
        filled=True,
        rounded=True,
        fontsize=TREE_FONT_SIZE,
    )
    plt.title("Decision Tree from Random Forest")
    plt.show()

# Tardis model main program:

In [19]:
def main_model_training() -> int:
    csv = csv_initialization("dataset.csv", separator=";")
    file = open("model.pkl", "w+b")
    if csv is None or file is None:
        return EPITECH_FAILURE
    features = csv.iloc[:, :-1].values
    targets = csv.iloc[:, -1:].values
    features_train, features_test, targets_train, targets_test = split_the_dataset(
        csv, features, targets
    )
    regressor = ExtraTreesRegressor(
        n_estimators=NBR_DECISION_TREES,
        max_features="sqrt",
        max_depth=15,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
        oob_score=True,
        bootstrap=True,
    )
    automatic_training(
        features_train, targets_train, features_test, targets_test, regressor
    )
    # show_model_performances(features, targets, regressor)
    dump(regressor, file=file)
    file.close()
    if SHOW:
        get_tree(csv, regressor)
    return EPITECH_SUCCESS


if __name__ == "__main__":
    main_model_training()

/home/alexisguibert/Bureau/Work/G2/G2 - Data Analysis/tardis-code_breakers/venv-tardis/lib/python3.12/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Out-of-Bag Score: 0.7613728962306376
Mean Absolute Error: 0.6175434451549543
Mean Squared Error: 0.634005803285581
R-squared: 0.7710133914637671
